# 🏛️ NexCiv AI: 02. Data Validation & Group Splitting
### Schema Verification, Zero-PII Audit & Leak-Free Parallel Group Splitting
---
### 📌 Critical Rule:
> **If we create parallel translations, all three versions (te, en, te_latn) must remain in the same dataset split.**
> Otherwise, the model will memorize translations rather than learning civic semantics.
> We enforce this via `GroupShuffleSplit` on `parallel_group_id`.

In [ ]:
import re
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

csv_path = '../../nexciv_multilingual_complaints_v1.csv'
if not os.path.exists(csv_path):
    csv_path = 'nexciv_multilingual_complaints_v1.csv'

df = pd.read_csv(csv_path)

# 1. Required Columns Assertion
required = [
    'complaint_id', 'parallel_group_id', 'complaint_text', 'language_style',
    'department', 'complaint_type', 'source_type', 'verified'
]
assert all(col in df.columns for col in required), 'Missing required schema columns!'
df = df.dropna(subset=['complaint_text', 'department', 'complaint_type'])
assert df['complaint_id'].is_unique, 'Duplicate complaint_id detected!'
print("✅ Schema assertion passed. Total valid rows:", len(df))

# 2. Zero-PII Audit
for text in df['complaint_text']:
    assert not re.search(r'\b[6-9]\d{9}\b', text), f"Phone number found in: {text}"
    assert '@' not in text, f"Email address found in: {text}"
print("✅ Zero-PII Audit passed: No phone numbers, emails, or personal identifiers.")

# 3. GroupShuffleSplit by parallel_group_id (70% Train, 15% Val, 15% Test)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df['parallel_group_id']))
train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['parallel_group_id']))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Check zero leakage across groups
assert len(set(train_df['parallel_group_id']).intersection(set(val_df['parallel_group_id']))) == 0
assert len(set(train_df['parallel_group_id']).intersection(set(test_df['parallel_group_id']))) == 0
assert len(set(val_df['parallel_group_id']).intersection(set(test_df['parallel_group_id']))) == 0
print(f"✅ Zero Leakage Confirmed! Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")